# LangGraph third-party agent

This notebook is a **LangGraph ReAct agent** calling CDP Agent Gateway the way a partner SDK would: POST JSON-RPC to `/mcp/spark`, `/mcp/hive`, and `/mcp/impala` with a Knox JWT. It never talks to Knox, Livy, HiveServer2, or Impala directly, and it does not use Streamable HTTP.

The hardcoded Spark → Hive walkthrough stays in [`third_party_agent.ipynb`](third_party_agent.ipynb). This notebook tests the agent-framework path: discover tools, bind them, let the graph call them.

The install cell matches the Workbench LangChain line: CML `langchain` 0.2.x keeps `langchain-core` 0.2 and LangGraph 0.2 (so `langchain-aws` / `langchain-community` stay satisfied). CML `langchain` 0.3.x uses LangGraph 0.3. It must not install langchain-core 1.x. If a previous cell installed 0.3 on a 0.2 runtime (or 1.x on any runtime), restart the session and re-run.

**Session secrets (not git, not AMP project env):**

- Knox JWT — paste in the token cell (`getpass`). Optional: `KNOX_TOKEN` for this engine.
- Model endpoint — form for **Model URL**, **Model ID**, and **Model token** (OpenAI-compatible, this engine only). Optional cloud keys: `OPENAI_API_KEY` / `ANTHROPIC_API_KEY`.

Do not print either secret. Compose MCP also sends `X-Agent-Key` (default `lab-agent`).

In [ ]:
import os
import sys
from pathlib import Path

root = Path(os.environ.get("AGENTGATEWAY_ROOT") or Path.cwd())
if not (root / "pyproject.toml").is_file():
    alt = Path("/home/cdsw")
    if (alt / "pyproject.toml").is_file():
        root = alt
agent_dir = root / "examples" / "agent"
src = root / "src"
for path in (agent_dir, src):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from langgraph_mcp import (
    apply_model_form,
    apply_model_settings,
    chat_model,
    install_langgraph_deps,
    invoke_agent,
    langchain_tools,
    last_ai_text,
    make_agent,
    show_model_form,
    tool_names_used,
)
from mcp_agent import knox_token_status, load_knox_token, mcp_base_url, profile

install_langgraph_deps(root=root)

print("profile:", profile())
print("spark url:", mcp_base_url("spark"))
print("hive url:", mcp_base_url("hive"))
print("impala url:", mcp_base_url("impala"))

## Knox JWT (this session only)

AMP project env is for `KNOX_PROXY_URL`, not the user bearer. Paste a **Knox Token API JWT** (three segments, usually starts with `eyJ`). Do not paste a Knox passcode, cookie, or `Bearer ` prefix.

The cell stores it in `os.environ` for this engine only and never prints the bearer. It does print `alg`, `iss`, `sub`, and seconds until `exp`. `401 invalid_token` means the gateway could not parse the value as a JWT.

In [ ]:
token = load_knox_token(prompt=True, ignore_non_jwt_env=True)
status = knox_token_status(token)
print("knox jwt:", "set" if status["set"] else "missing")
print("jwt_shaped:", status["jwt_shaped"])
print("alg:", status["alg"] or "-")
print("iss:", status["iss"] or "-")
print("sub:", status["sub"] or "-")
print("exp_in_s:", status["exp_in_s"])
print("hint:", status["hint"])
if not status["jwt_shaped"] or status["hint"] != "ok":
    raise RuntimeError(status["hint"] or "Knox JWT is not usable")

## Model endpoint (this session only)

LangGraph needs an OpenAI-compatible chat model. Fill **Model URL**, **Model ID**, and **Model token** (Cloudera AI Inference, Model Mesh, vLLM, OpenAI, etc.). The token is a password field and is never printed. Do not put these in AMP project env. The Knox JWT above is a different secret.

## Model endpoint (this session only)

LangGraph needs an OpenAI-compatible chat model. Fill **Model URL**, **Model ID**, and **Model token** (Cloudera AI Inference, Model Mesh, vLLM, OpenAI, etc.). The token is a password field and is never printed. Do not put these in AMP project env. The Knox JWT above is a different secret.

URL is typically the `/v1` base (Chat Completions). Then run the apply cell.

vLLM without `--enable-auto-tool-choice` still treats OpenAI `tools` as `tool_choice=auto` (omitting the field is not enough). For a custom Model URL this notebook sends **prompt-parsed** tool calls instead (JSON `{"name", "arguments"}`, no `tools` in the HTTP body). Re-run **make_agent** after pulling that helper. Set `MODEL_TOOL_CHOICE=auto` only if the server already has `--enable-auto-tool-choice --tool-call-parser`.


In [ ]:
from getpass import getpass

form = show_model_form()
if form is None:
    print("ipywidgets unavailable; prompts below (token not echoed)")
    apply_model_settings(
        url=input("Model URL: "),
        model_id=input("Model ID: "),
        token=getpass("Model token (not echoed): "),
    )
else:
    print("fill Model URL, Model ID, and Model token, then run the next cell")


In [ ]:
status = apply_model_form(globals().get("form"))
print("model url:", status["url"] or "(cloud default)")
print("model id:", status["model_id"] or "(cloud default)")
print("model token:", "set" if status["token_set"] else "missing")
print("hint:", status.get("hint") or "ok")


## Bind MCP tools

`tools/list` on each adapter becomes LangChain tools. Adapters AMP returns as `adapter_disabled` (often Impala) are skipped. The ReAct graph then calls `tools/call` with the same Knox JWT. This is not `langchain-mcp-adapters` Streamable HTTP.

In [ ]:
tools = langchain_tools()
print("bound:", ", ".join(sorted(t.name for t in tools)))
agent = make_agent(chat_model(), tools=tools)
print("graph:", type(agent).__name__)

## Read-only agent turn

Ask for catalogs the Knox subject can see. This is the default third-party test: no `spark_submit_batch`, so it is safe against a live cluster and finishes quickly.

Override the prompt with `LANGGRAPH_QUESTION`.

In [ ]:
question = (os.environ.get("LANGGRAPH_QUESTION") or "").strip() or (
    "List Spark MCP batches for this Knox user, then list Hive databases Ranger allows. "
    "Summarize tool names you used. Do not submit a Spark job."
)
print("question:", question)
result = invoke_agent(question, agent=agent)
print("tools used:", ", ".join(tool_names_used(result)) or "(none)")
print(last_ai_text(result))

## Optional write path

Set `LANGGRAPH_RUN_SUBMIT=1` only after the job file is on a Ranger-allowed URI (`gateway webhdfs put` on Compose, or `SPARK_FILE_URI` on AMP). The agent may call `spark_submit_batch` (a write as the Knox subject) and poll `spark_get_batch`. That can take several minutes and counts against MCP burst + daily submit quota.

For a deterministic submit → Hive select, use [`third_party_agent.ipynb`](third_party_agent.ipynb) instead.

In [ ]:
if os.environ.get("LANGGRAPH_RUN_SUBMIT", "").strip() not in {"1", "true", "yes"}:
    print("skip submit (set LANGGRAPH_RUN_SUBMIT=1 to enable)")
else:
    from mcp_agent import knox_user_from_spark, spark_job_uri, submit_spark_example

    knox_user = knox_user_from_spark()
    file_uri = spark_job_uri(knox_user)
    database = knox_user.split("@", 1)[0]
    submit = submit_spark_example(file_uri=file_uri, name="count-to-10")
    submit_question = (
        f"Batch id {submit.get('id')} name count-to-10 is already submitted "
        f"(reused={submit.get('reused')}). Do not call spark_submit_batch again. "
        f"Poll spark_get_batch until success or dead. "
        f"If success, hive_select database={database} table=count_to_10 columns n limit 10. "
        "Do not invent SQL."
    )
    print("knox_user:", knox_user)
    print("file:", file_uri)
    submit_result = invoke_agent(submit_question, agent=agent, recursion_limit=50)
    print("tools used:", ", ".join(tool_names_used(submit_result)) or "(none)")
    print(last_ai_text(submit_result))